# 3.2 — Data selection, filtering, and Boolean logic

Translate an analysis question into displayed columns and row conditions, expressed as verifiable Boolean masks.

## Introduction

Use this Notebook to verify the lesson concepts with actual data and code.

## Learning outcomes

- Decompose an analysis question into displayed columns and row conditions.
- Create Boolean masks from comparisons and combine them correctly.
- Represent membership, ranges, and missingness explicitly.
- Verify a filtered result through counts and deterministic ordering.

> **Learning route:** Required: 3.2.1–3.2.6 / Integrated practice: 3.2.7


## 3.2.1 Separate a question into displayed columns and row conditions

For “show centre names in February and March with at least 30 registrations and attendance below 80%,” separate display columns, month membership, a registration lower bound, and an attendance-rate upper bound. Confirm required columns before filtering.

In [ ]:
from pathlib import Path
import pandas as pd


def find_course_data(filename):
    """Find course data without depending on the Notebook start directory."""
    roots = [Path.cwd(), *Path.cwd().parents, Path.home() / "work", Path("/opt/python-lab/course-materials")]
    checked = []
    for root in roots:
        for candidate in (root / "data" / filename, root / filename):
            candidate = candidate.expanduser()
            if candidate in checked:
                continue
            checked.append(candidate)
            if candidate.is_file():
                return candidate
    locations = "\n".join(f"- {path}" for path in checked)
    raise FileNotFoundError(f"Course data file {filename!r} was not found. Checked:\n{locations}")


data_file = find_course_data("learning-centres-practice.csv")
print("Loading:", data_file.resolve())
df = pd.read_csv(data_file, encoding="utf-8", dtype={"centre_id": "string", "month": "string"})
print("Shape:", df.shape)


### Select the fields needed for the result

One column is a Series; several columns form a DataFrame selected with a list of names. A missing name raises `KeyError`, so compare required names with `df.columns` first.

In [ ]:
required = {"month", "centre_name", "registered", "attended", "completed"}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"Missing required columns: {sorted(missing)}")

focused = df[["month", "centre_name", "registered", "attended", "completed"]]
print(focused.head(3))


## 3.2.2 Select rows and columns with labels and positions

`loc[row condition, column names]` uses index labels and named columns, keeping analytical meaning in the code. `iloc[row positions, column positions]` uses zero-based positions and is useful for checks such as the first few rows. Index labels are not guaranteed to be 0, 1, 2, so do not confuse the two.

In [ ]:
print(df.loc[df.index[:2], ["centre_id", "centre_name"]])
print(df.iloc[:2, [1, 2]])


## 3.2.3 Turn comparisons into Boolean masks

Comparing a column with a value creates a Boolean Series aligned with the source index. Only `True` rows remain. `mask.sum()` counts matching rows. Print counts before and after selection to expose mistaken conditions.

In [ ]:
large = df["registered"] >= 30
print(large.head())
print("Matching rows:", int(large.sum()))
print(df.loc[large, ["month", "centre_name", "registered"]].head())


## 3.2.4 Build compound conditions with Boolean logic

Use Python `and`, `or`, and `not` for individual Boolean values. Combine pandas Series row by row with `&`, `|`, and `~`, placing each comparison in parentheses. Passing Series to `and` asks for one truth value for the whole Series and raises an error.

In [ ]:
truth = pd.DataFrame({"A": [False, False, True, True], "B": [False, True, False, True]})
truth["A & B"] = truth["A"] & truth["B"]
truth["A | B"] = truth["A"] | truth["B"]
truth["~A"] = ~truth["A"]
truth


### Verify AND, OR, and NOT through counts

AND narrows a condition, while OR usually broadens it. NOT reverses True and False. By De Morgan's law, `~(A | B)` equals `(~A) & (~B)`, although the form that best communicates the operational meaning is preferable.

In [ ]:
report = df.assign(
    attendance_rate=df["attended"] / df["registered"] * 100,
    completion_rate=df["completed"] / df["registered"] * 100,
)
large = report["registered"] >= 30
low_attendance = report["attendance_rate"] < 80
print("AND count:", int((large & low_attendance).sum()))
print("OR count:", int((large | low_attendance).sum()))


## 3.2.5 Express membership, ranges, and missingness

Use `isin()` for membership in several candidate values and `between()` for lower and upper bounds. `between()` includes both endpoints by default, so match boundary inclusion to the wording of the question.

In [ ]:
months = report["month"].isin(["2026-02", "2026-03"])
medium_size = report["registered"].between(25, 35, inclusive="both")
print(report.loc[months & medium_size, ["month", "centre_name", "registered"]])


### Do not lose missing rows accidentally

Comparisons with missing values are usually False, so rows can disappear without explaining why. Add `notna()` when a value is required or `isna()` when investigating missingness, and record the count separately. Do not fill values here; retain them as quality issues for Lesson 3.3.

In [ ]:
has_attendance = report["attended"].notna()
low_attendance_known = has_attendance & (report["attendance_rate"] < 80)
print("Known attendance below 80%:", int(low_attendance_known.sum()))
print("Missing attendance:", int(report["attended"].isna().sum()))


## 3.2.6 Filter, order, and verify the result

Build named masks for each part of the question, verify them separately, then combine them in `loc`. Do not call a subset “all data”; record the source count, selected count, and condition.

In [ ]:
selected_columns = ["month", "centre_id", "centre_name", "registered", "attendance_rate", "completion_rate"]
priority_mask = (
    report["month"].isin(["2026-02", "2026-03"])
    & (report["registered"] >= 30)
    & report["attended"].notna()
    & (report["attendance_rate"] < 80)
)
priority = report.loc[priority_mask, selected_columns].sort_values(["month", "centre_id"])
print("Source rows:", len(report), "selected rows:", len(priority))
priority


### Align masks and rows by index label

pandas aligns a Boolean mask by index label, not only by physical position. Reusing a mask from another DataFrame or from before an index change can cause misalignment or an error. Normally build a mask from the same DataFrame being selected.

## 3.2.7 Integrated practice: translate and verify another question

Select centres in February or March, for Python Foundations, with 25–40 registrations, completion below 75%, and a known completion value. Validate required columns, print counts for each partial and final mask, and sort by month and centre ID. Create and compare a second question containing one negated condition.

In [ ]:
# Write the transfer solution here.


## Summary

- Separated an analysis question into output columns and named partial conditions.
- Combined Boolean masks and selected rows and columns with loc.
- Made boundaries and missingness explicit, then verified counts and order.

## Next

The selection is reproducible. Lesson 3.3 handles missingness, inconsistent labels, contradictions, and duplicates while preserving evidence.

**Estimated learning time:** about 4 hours
